# 17쪽 · 로컬 서버와 SDK

## 준비
아래 셀은 파일 경로와 `.env` 로더를 준비합니다. 한 번 실행하고 넘어가세요.

In [ ]:
from pathlib import Path
import sys

# 노트북을 연 위치가 ipynb 폴더이면 한 단계 위로 이동합니다.
ROOT = Path.cwd()

if ROOT.name == "ipynb":
    ROOT = ROOT.parent

if not (ROOT / "lab" / "config.py").exists():
    raise RuntimeError("Langgraph_6_Week_Code 또는 ipynb 폴더에서 실행하세요.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lab.config import make_llm, load_settings

## 1. 별도 터미널에서 서버 시작

가상환경을 활성화하고 `Langgraph_6_Week_Code` 폴더에서 실행하세요.

```bash
python -m pip install -r requirements-studio.txt
langgraph dev
```

이 터미널은 켜둡니다. Studio는 서버가 출력한 URL로 엽니다.

## 2. 서버 연결

URL은 .env에서 읽습니다. 서버가 없으면 다음 셀의 요청이 실패합니다.

In [ ]:
from langgraph_sdk import get_client

settings = load_settings()
client = get_client(url=settings.server_url)

## 3. Assistant 조회

서버에 등록된 assistant를 확인합니다.

In [ ]:
assistants = await client.assistants.search()

for assistant in assistants:
    print(assistant["graph_id"])

**결과 해설**

`agent`가 표시되어야 합니다.

## 4. 새 대화 생성

대화 번호를 저장합니다.

In [ ]:
thread = await client.threads.create()
thread_id = thread["thread_id"]

print(thread_id)

## 5. 첫 질문 실행

각 단계의 마지막 메시지를 출력합니다.

In [ ]:
input_data = {
    "messages": [
        {"role": "user", "content": "3과 4를 더해줘."},
    ],
}

async for chunk in client.runs.stream(
    thread_id,
    "agent",
    input=input_data,
    stream_mode="values",
):
    if chunk.event == "error":
        raise RuntimeError(chunk.data)

    if chunk.event == "values" and chunk.data:
        messages = chunk.data.get("messages", [])

        if messages:
            print(messages[-1])

**결과 해설**

도구 요청, 계산 결과, 최종 답변을 확인합니다.

## 6. 같은 대화에 두 번째 질문

thread_id를 바꾸지 않습니다.

In [ ]:
input_data = {
    "messages": [
        {"role": "user", "content": "그 결과에 2를 곱해줘."},
    ],
}

async for chunk in client.runs.stream(
    thread_id,
    "agent",
    input=input_data,
    stream_mode="values",
):
    if chunk.event == "error":
        raise RuntimeError(chunk.data)

    if chunk.event == "values" and chunk.data:
        messages = chunk.data.get("messages", [])

        if messages:
            print(messages[-1])

**결과 해설**

이전 결과 7을 참고하여 14를 계산할 수 있습니다. 서버가 상태 저장을 관리합니다. values는 토큰이 아니라 단계별 state입니다.